In [124]:
# -*- coding: utf-8 -*-
"""
End-to-end LULC -> 0.1° fractions (conservative/area-weighted) -> PFT fractions via CW-table.
Inputs:
  - Single-band classified GeoTIFF (integer classes) in any projected CRS.
    Example: reclass2019.tif with codes:
        1=Water, 3=Bog, 5=Coniferous forest, 7=Open crown forest,
        9=Deciduous tree, 11=Marsh, 13=Built up, 15=Fen, 17=Mixed forest
Outputs (CSV, in OUT_DIR):
  - LULC_fractions_0p1deg_conservative_mappedvalid.csv
  - PFT_fractions_0p1deg_tidy.csv
  - PFT_fractions_0p1deg_wide.csv             (rows sum to 1)
  - PFT_fractions_0p1deg_wide_with_LULC.csv   (adds LULC coverage columns & summaries)
  - latlon_to_LULC_list.csv                   (lightweight lookup for each 0.1° cell)
Install:
  pip install rasterio numpy pandas affine
"""

'\nEnd-to-end LULC -> 0.1° fractions (conservative/area-weighted) -> PFT fractions via CW-table.\nInputs:\n  - Single-band classified GeoTIFF (integer classes) in any projected CRS.\n    Example: reclass2019.tif with codes:\n        1=Water, 3=Bog, 5=Coniferous forest, 7=Open crown forest,\n        9=Deciduous tree, 11=Marsh, 13=Built up, 15=Fen, 17=Mixed forest\nOutputs (CSV, in OUT_DIR):\n  - LULC_fractions_0p1deg_conservative_mappedvalid.csv\n  - PFT_fractions_0p1deg_tidy.csv\n  - PFT_fractions_0p1deg_wide.csv             (rows sum to 1)\n  - PFT_fractions_0p1deg_wide_with_LULC.csv   (adds LULC coverage columns & summaries)\n  - latlon_to_LULC_list.csv                   (lightweight lookup for each 0.1° cell)\nInstall:\n  pip install rasterio numpy pandas affine\n'

In [213]:
import math
from pathlib import Path
from typing import Dict, List
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_bounds
from affine import Affine

In [281]:
# ---------- USER SETTINGS ----------
TIFF_PATH = r"D:/Reclassified/Maps/reclass2014.tif"       # path to your single-band classified GeoTIFF
OUT_DIR = Path(r"D:/Reclassified/PFTs csv/2014")  # output folder
GRID_RES_DEG = 0.05                 # 0.1° target grid

In [282]:
# LULC codes -> names (as provided)
LULC_MAP: Dict[int, str] = {
    1:  "Water",
    3:  "Bog",
    5:  "Coniferous forest",
    7:  "Open crown forest",
    9:  "Deciduous tree",
    11: "Marsh",
    13: "Builtup"
    15: "Fen",
    17: "Mixed forest",
}

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2333431966.py, line 9)

In [283]:
# CW-table (rows sum to 1). PFT column names preserved in outputs.
PFTS: List[str] = [
    "pft1_NLE","pft2_NLD","pft3_BLE","pft45_BCD_BDD","pft67_C3C_C4C",
    "pft89_C3G_C4G","pft10_Sedge","pft11_SBE","pft12_SBD","Urban","Lake","Ocean","Bare"
]
CW_WEIGHTS: Dict[str, List[float]] = {
    "Deciduous tree":   [0,    0,    0,   0.70, 0,    0.25, 0,    0,    0,    0,    0,    0,   0.05],
    "Coniferous tree":  [0.80, 0,    0,   0,    0,    0.10, 0,    0.05, 0,    0,    0,    0,   0.05],
    "Open crown forest":[0.35, 0,    0,   0.50, 0,    0.05, 0,    0.10, 0,    0,    0,    0,   0.00],
    "Mixed forest":     [0,    0,    0,   0.20, 0,    0.25, 0,    0.40, 0,    0,    0,    0,   0.15],
    "Fen, Bog, Marsh":  [0,    0,    0,   0,    0,    0.20, 0.10, 0.20, 0.25, 0,    0.15, 0,   0.10],
    "Urban areas":      [0,    0,    0,   0.05, 0,    0.15, 0,    0,    0,    0.75, 0.05, 0,   0.00],
    "Water":            [0,    0,    0,   0,    0,    0,    0,    0,    0,    0,    1.00, 0,   0.00],
}

In [284]:

def cw_key(name: str) -> str:
    n = str(name).strip().lower()
    if n in ("deciduous tree",): return "Deciduous tree"
    if n in ("coniferous tree","coniferous forest"): return "Coniferous tree"
    if n in ("open crown forest",): return "Open crown forest"
    if n in ("mixed forest",): return "Mixed forest"
    #if n in ("built up","urban","urban areas","urban area"): return "Urban areas"
    if n in ("water","lake","lakes"): return "Water"
    if n in ("fen","bog","marsh","fen, bog, marsh"): return "Fen, Bog, Marsh"
    return str(name)

def snap_floor(v, res): return math.floor(v/res)*res
def snap_ceil(v, res):  return math.ceil(v/res)*res

def cell_area_km2(lat0_deg, dlat_deg, dlon_deg):
    R = 6371.0088
    lat1 = math.radians(lat0_deg)
    lat2 = math.radians(lat0_deg + dlat_deg)
    dlon = math.radians(dlon_deg)
    return (R**2) * abs(math.sin(lat2)-math.sin(lat1)) * abs(dlon)

def compute_lulc_fractions_from_tiff(tif_path: Path) -> pd.DataFrame:
    with rasterio.open(tif_path) as src:
        if src.count != 1:
            raise ValueError("Input raster must be single-band with integer class codes.")
        src_crs, src_transform = src.crs, src.transform
        H, W = src.height, src.width
        cls = src.read(1)
        nodata = src.nodata
        wgs84_bounds = transform_bounds(src_crs, "EPSG:4326", *src.bounds, densify_pts=21)

In [285]:
    min_lon = max(-180.0, snap_floor(wgs84_bounds[0], GRID_RES_DEG))
    min_lat = max(-90.0,  snap_floor(wgs84_bounds[1], GRID_RES_DEG))
    max_lon = min( 180.0, snap_ceil(wgs84_bounds[2], GRID_RES_DEG))
    max_lat = min(  90.0, snap_ceil(wgs84_bounds[3], GRID_RES_DEG))
    width  = int(round((max_lon-min_lon)/GRID_RES_DEG))
    height = int(round((max_lat-min_lat)/GRID_RES_DEG))

In [286]:
dst_transform = Affine(GRID_RES_DEG, 0.0, min_lon, 0.0, -GRID_RES_DEG, max_lat)

codes_present = sorted(set(np.unique(cls)).intersection(LULC_MAP.keys()))
if nodata is not None:
    try: codes_present = [c for c in codes_present if c != int(nodata)]
    except Exception: pass
if not codes_present:
    raise RuntimeError("No overlapping LULC codes between raster and LULC_MAP.")

In [287]:
# conservative remap per class
fractions = {}
mapped_valid = np.zeros((height, width), dtype=np.float32)
for code in codes_present:
    src_mask = (cls == code).astype(np.float32)
    dst = np.zeros((height, width), dtype=np.float32)
    reproject(src_mask, dst,
                src_transform=src_transform, src_crs=src_crs,
                dst_transform=dst_transform, dst_crs="EPSG:4326",
                dst_nodata=0.0, resampling=Resampling.average)
    fractions[code] = dst
    mapped_valid += dst
np.clip(mapped_valid, 0.0, 1.0, out=mapped_valid)

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [288]:
# build tidy df
rows = []
oid = 1
for y in range(height):
    lat0 = max_lat - (y+1)*GRID_RES_DEG
    area_cell = cell_area_km2(lat0, GRID_RES_DEG, GRID_RES_DEG)
    for x in range(width):
        lon0 = min_lon + x*GRID_RES_DEG
        mv = float(mapped_valid[y,x])
        if mv <= 0: continue
        for code in codes_present:
            f_full = float(fractions[code][y,x])
            if f_full <= 0: continue
            rows.append({
                    "object_id": oid,
                    "cell_id": y*width + x,
                    "grid_cell": f"{lon0:.1f}_{lat0:.1f}",
                    "lon0": lon0, "lat0": lat0,
                    "lulc_code": int(code),
                    "classic_tile": LULC_MAP[int(code)],
                    "fraction_of_cell": f_full,
                    "relative_area":   f_full / mv,
                    "area_km2":        f_full * area_cell,
                    "cell_area_km2":   area_cell,
                    "mapped_valid_frac": mv
                })
            oid += 1
df_lulc = pd.DataFrame(rows)

In [289]:
# ---------- load or compute df_lulc ----------
OUT_DIR.mkdir(parents=True, exist_ok=True)
lulc_csv_path = OUT_DIR / "LULC_fractions_0p1deg_conservative_mappedvalid.csv"

need_compute = True
if "df_lulc" in globals() and isinstance(globals()["df_lulc"], pd.DataFrame) and not globals()["df_lulc"].empty:
    df_lulc = globals()["df_lulc"].copy()
    need_compute = False
elif lulc_csv_path.exists():
    df_lulc = pd.read_csv(lulc_csv_path)
    need_compute = False

if need_compute:
    # compute from TIFF (only if not already loaded/saved)
    df_lulc = compute_lulc_fractions_from_tiff(TIFF_PATH)
    df_lulc.to_csv(lulc_csv_path, index=False)

In [290]:
# --- make sure LULC name column exists ---
if "classic_tile" not in df_lulc.columns:
    if "lulc_code" in df_lulc.columns:
        df_lulc["classic_tile"] = df_lulc["lulc_code"].map(LULC_MAP).astype(str)
    else:
        raise KeyError("df_lulc has no 'classic_tile' or 'lulc_code'. Columns: "
                       f"{list(df_lulc.columns)}")

In [291]:
# ---------- apply CW -> PFT tidy & wide ----------
weights = (
    pd.DataFrame.from_dict(CW_WEIGHTS, orient="index", columns=PFTS)
      .reset_index().rename(columns={"index":"cw_group"})
)
df_lulc["cw_group"] = df_lulc["classic_tile"].astype(str).apply(cw_key)
dfm = df_lulc.merge(weights, on="cw_group", how="left")

In [292]:
# aggregate to tidy pft
parts = []
gb = dfm.groupby(["cell_id","grid_cell","lon0","lat0"], sort=False)
for p in PFTS:
    tmp = gb.apply(lambda g: pd.Series({
        "fraction_relative": (g["relative_area"]   * g[p]).sum(),
        "fraction_fullcell": (g["fraction_of_cell"]* g[p]).sum(),
        "area_km2":          (g["area_km2"]        * g[p]).sum(),
        "cell_area_km2":      g["cell_area_km2"].iloc[0],
        "mapped_valid_frac":  g["mapped_valid_frac"].iloc[0],
    })).reset_index()
    tmp["pft"] = p
    parts.append(tmp)
pft_tidy = pd.concat(parts, ignore_index=True)

C:\Users\momu1064\AppData\Local\Temp\ipykernel_22868\593729658.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tmp = gb.apply(lambda g: pd.Series({
C:\Users\momu1064\AppData\Local\Temp\ipykernel_22868\593729658.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tmp = gb.apply(lambda g: pd.Series({
C:\Users\momu1064\AppData\Local\Temp\ipykernel_22868\593729658.py:5: DeprecationWarning: DataFrameGroupBy.

In [293]:
# normalize per cell (safety against fp drift)
row_sums = pft_tidy.groupby("cell_id", sort=False)["fraction_relative"].transform("sum")
pft_tidy["fraction_relative"] = np.where(row_sums>0, pft_tidy["fraction_relative"]/row_sums, 0.0)

In [294]:
# wide (relative fractions)
pft_wide = (
    pft_tidy.pivot_table(index=["cell_id","grid_cell","lon0","lat0"],
                         columns="pft", values="fraction_relative", aggfunc="first")
            .reset_index()
)
pft_cols = [c for c in pft_wide.columns if c in PFTS]
pft_wide["_row_sum"] = pft_wide[pft_cols].sum(axis=1)  # QA: should be ~1.0

In [295]:
# ---------- enrich PFT wide with LULC coverage per cell ----------
def safe_label(x: str) -> str:
    return ("LULC_" + str(x).strip().replace(" ", "_").replace(",", "").replace("/", "_").replace("-", "_"))

lulc_rel = (df_lulc.groupby(["cell_id","classic_tile"], as_index=False)["relative_area"].sum())
lulc_wide = (lulc_rel.pivot_table(index="cell_id", columns="classic_tile", values="relative_area", fill_value=0.0)
             .rename(columns=safe_label).reset_index())

def summarize_cell(g: pd.DataFrame) -> pd.Series:
    g = g.sort_values("relative_area", ascending=False)
    present = "; ".join(g.loc[g["relative_area"]>0, "classic_tile"].astype(str).unique())
    mix = " | ".join(f"{r.classic_tile}:{r.relative_area:.3f}" for r in g.itertuples())
    top = g.iloc[0]["classic_tile"] if len(g) else ""
    topf = g.iloc[0]["relative_area"] if len(g) else 0.0
    return pd.Series({"lulc_present": present, "lulc_mix": mix, "lulc_top": top, "lulc_top_frac": topf})

lulc_summary = df_lulc.groupby("cell_id", sort=False).apply(summarize_cell).reset_index()

enriched = (pft_wide.merge(lulc_wide, on="cell_id", how="left")
                     .merge(lulc_summary, on="cell_id", how="left"))

C:\Users\momu1064\AppData\Local\Temp\ipykernel_22868\3960361944.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  lulc_summary = df_lulc.groupby("cell_id", sort=False).apply(summarize_cell).reset_index()


In [296]:
# ---------- write outputs ----------
pft_tidy_out = OUT_DIR / "PFT_fractions_0.05deg_tidy.csv"
pft_wide_out = OUT_DIR / "PFT_fractions_0.05deg_wide.csv"
enriched_out = OUT_DIR / "PFT_fractions_0.05deg_wide_with_LULC.csv"
latlon_list_out = OUT_DIR / "latlon_to_LULC_list0.05deg.csv"

pft_tidy.to_csv(pft_tidy_out, index=False)
pft_wide.to_csv(pft_wide_out, index=False)
enriched.to_csv(enriched_out, index=False)
enriched[["lon0","lat0","lulc_present","lulc_mix","lulc_top","lulc_top_frac"]].to_csv(latlon_list_out, index=False)

print("Done.")
print("df_lulc:", df_lulc.shape, "columns:", list(df_lulc.columns))
print("pft_wide row-sum (min,max):", float(pft_wide['_row_sum'].min()), float(pft_wide['_row_sum'].max()))
print("Outputs ->", OUT_DIR.resolve())
for p in [lulc_csv_path, pft_tidy_out, pft_wide_out, enriched_out, latlon_list_out]:
    print(" -", p.name)

Done.
df_lulc: (4933, 13) columns: ['object_id', 'cell_id', 'grid_cell', 'lon0', 'lat0', 'lulc_code', 'classic_tile', 'fraction_of_cell', 'relative_area', 'area_km2', 'cell_area_km2', 'mapped_valid_frac', 'cw_group']
pft_wide row-sum (min,max): 0.9999999999999998 1.0000000000000002
Outputs -> D:\Reclassified\PFTs csv\2014
 - LULC_fractions_0p1deg_conservative_mappedvalid.csv
 - PFT_fractions_0.05deg_tidy.csv
 - PFT_fractions_0.05deg_wide.csv
 - PFT_fractions_0.05deg_wide_with_LULC.csv
 - latlon_to_LULC_list0.05deg.csv
